<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_5_1_Trees_Data_and_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Trees: The Foundation of Ensembles

Author: Brad Sheese

---

## Introduction
In this notebook, we move from the probability-based logic of Logistic Regression to the "flowchart" logic of **Decision Trees**. Decision trees are highly interpretable, making them the most easily explainable machine learning models for non-technical audiences.

However, we will also see their primary weakness: **overfitting**. This notebook will serve as our to building building high-performance ensembles like Random Forests and Boosting models later in this series.

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Load and Explore** the Wisconsin Breast Cancer (Diagnostic) dataset.
2. **Implement** a Decision Tree classifier using `scikit-learn`.
3. **Visualize** the model's decision-making process with a tree diagram.
4. **Evaluate** the impact of tree depth on model accuracy and overfitting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set style
sns.set_context("talk")
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Preparation: Wisconsin Breast Cancer Dataset

We are working with measurements taken from digitized images of a fine needle aspirate (FNA) of a breast mass. These describe characteristics of the cell nuclei present in the image.

### Data Details
- Name: Wisconsin Diagnostic Breast Cancer (WDBC)
- Alternative Name: Breast Cancer Wisconsin (Diagnostic) Data Set
- Source: UCI Machine Learning Repository
- Size: 569 samples
- Features: 30 real-valued features computed from digitized images of fine needle aspirate (FNA) of breast masses
- Feature Groups: For each of 10 morphological characteristics, three values are computed:
  - Mean
  - Standard error
  - "Worst" (mean of the three largest values)
- The 10 base features:
  1. Radius (mean of distances from center to perimeter points)
  2. Texture (standard deviation of gray-scale values)
  3. Perimeter
  4. Area
  5. Smoothness (local variation in radius lengths)
  6. Compactness (perimeter²/area - 1.0)
  7. Concavity (severity of concave portions of contour)
  8. Concave points (number of concave portions of contour)
  9. Symmetry
  10. Fractal dimension ("coastline approximation" - 1)
- Total Features: 30 (3 feature groups X 10 base features)
- Usage: Features describe characteristics of cell nuclei present in the image


**Goal**: Predict whether a mass is **Malignant** (1) or **Benign** (0).

In [ ]:
# Load the dataset
data = load_breast_cancer(as_frame=True)
df = data.frame

print(f"Dataset Shape: {df.shape}")
print(f"Target Distribution:\n{df['target'].value_counts(normalize=True)}")
df.head()

### Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
# let's visualize our distributions
n_cols = 5
n_features = len(df.columns)
n_rows = (n_features + n_cols - 1) // n_cols # Calculate number of rows needed

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

for i, col in enumerate(df.columns):
    sns.histplot(df[col], ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_ylabel('') # Remove y-label to avoid clutter

# Turn off any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# this is very clean data...
# there's a lot of skew, but these distributions are
# better than many of the datasets we've been dealing with.
# Notably trees don't care about skew. So we don't need to worry about
# correction.

### Visualizing a Split
Decision trees find the best feature and "split point" to divide the data. Let's look at one feature, `mean concave points`, and see how it differs between Malignant and Benign tumors.

In [ ]:
sns.violinplot(x='target', y='mean concave points', data=df)
plt.title("Distribution of Mean Concave Points by Malignancy")
plt.xticks([0, 1], ['Benign', 'Malignant'])
plt.show()

This shows good differentiation. In fact, the tree models will, by themselves, identify this as a good starting point because of the clear differentiation you see here.

It's worth examining the other features in the same way to see they are not so good at differentiating.

In [ ]:
n_cols = 5
n_features = len(df.columns[:-1]) # Exclude 'target'
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(df.columns[:-1]):
    sns.violinplot(x='target', y=col, data=df, ax=axes[i])
    axes[i].set_title(f"{col}")
    axes[i].set_xticks([0, 1], ['Benign', 'Malignant'])
    axes[i].set_ylabel('') # Remove y-label to avoid clutter

# Turn off any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### Train/Test Split
To evaluate how our model will perform on brand-new data, we must split our data into training and testing sets.

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

---
## 2. The Building Block: A Single Decision Tree

A decision tree finds a specific feature and threshold to split the data. For example: "If `mean concave points` > 0.05, go left; otherwise, go right."

We use `DecisionTreeClassifier`. We will start by limiting the depth of the tree to `max_depth=3` to keep it interpretable.

Note: `clf` is a conventional abbreviation in scikit-learn and machine learning code that stands for "classifier". It's widely used in tutorials, documentation, and examples as a shorthand variable name for classification models.

In [ ]:
# Initialize and fit the model
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)


plt.figure(figsize=(20, 10))
plot_tree(clf,
          feature_names=data.feature_names,
          class_names=['Benign', 'Malignant'],
          filled=True,
          rounded=True)
plt.title("Decision Tree Flowchart (Max Depth = 3)")
plt.show()

### How to Read the Tree

The tree visualization above shows exactly how the model makes decisions. Each box (called a **node**) represents a step in the logic.

1.  **Condition (Top Line)**: This is the question the tree asks. For example, `mean concave points <= 0.051`.
    *   If **True**, go to the left child node.
    *   If **False**, go to the right child node.

2.  **gini**: This is a measure of **impurity** or disorder.
    *   A Gini of **0.0** means the node is **pure** (contains only one class).
    *   A Gini of **0.5** means the node is **maximally impure** (50% Benign, 50% Malignant).
    *   The goal of the tree is to reduce this number to 0.

3.  **samples**: The total number of patients in this node.
    *   Notice how the number of samples decreases as you go down the tree.

4.  **value**: The count of samples for each class: `[Benign count, Malignant count]`.
    *   E.g., `[200, 10]` means 200 Benign and 10 Malignant cases.

5.  **class**: The predicted class for this node, based on the majority vote in `value`.
    *   If `value = [200, 10]`, the class is **Benign**.

6.  **Color**:
    *   **Orange** indicates Benign (Class 0).
    *   **Blue** indicates Malignant (Class 1).
    *   The **darker** the color, the higher the purity (lower Gini score). White nodes are mixed.

---
## 3. The Overfitting Problem: Depth vs. Generalization

Decision trees have a major weakness: they love to memorize data. Or, slightly more formally, they are very good at overfitting your training set.

If we don't limit the depth (the number of levels), a decision tree will keep splitting until every single leaf node is pure (Gini = 0). While this guarantees 100% accuracy on the *training* data, it usually leads to overfitting.

As you hoefully remember by now, overfitting means the model has learned the specific noise and quirks of the training set rather than the general rules of breast cancer diagnosis. As a result, it likely performs poorly on new, unseen data (the test set).

Let's visualize this trade-off by training trees at different depths and comparing their performance.

In [ ]:
# Train trees with different depths and record accuracy
depths = range(1, 21)
train_scores = []
test_scores = []

for depth in depths:
    # Initialize and fit
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)

    # Record scores
    train_scores.append(accuracy_score(y_train, tree.predict(X_train)))
    test_scores.append(accuracy_score(y_test, tree.predict(X_test)))

# Plotting the Complexity Curve
plt.figure(figsize=(12, 6))
plt.plot(depths, train_scores, label='Training Accuracy', marker='o', linestyle='-')
plt.plot(depths, test_scores, label='Testing Accuracy', marker='o', linestyle='--')

plt.xlabel('Tree Depth (max_depth)')
plt.ylabel('Accuracy')
plt.title('Overfitting in Action: Training vs. Testing Accuracy')
plt.legend()
plt.grid(True)
plt.xticks(depths)
plt.show()

# Find the best test accuracy
best_depth = depths[np.argmax(test_scores)]
print(f"Best Testing Accuracy occurs at Depth: {best_depth}")
print(f"Training Accuracy at Depth {best_depth}: {train_scores[best_depth-1]:.3f}")
print(f"Testing Accuracy at Depth {best_depth}: {test_scores[best_depth-1]:.3f}")

### Analyzing the Complexity Curve

Look at the plot above:
1.  **Training Accuracy (Solid Line)**: Keeps going up as the tree gets deeper, eventually hitting 1.0 (100%). The model is perfectly memorizing the training data.
2.  **Testing Accuracy (Dashed Line)**: Increases initially as the model learns real patterns. However, after a certain depth (around 3-5), it plateaus or drops.

The gap between the two lines represents overfitting. The "sweet spot" is where the Testing Accuracy is highest before the lines diverge too much.

---
## 4. Robust Evaluation: K-Fold Cross-Validation

We just found the "best" depth using a single train/test split. But what if our test set just happened to be easy? Or unusually hard?

A single split can be misleading. To get a more reliable estimate of our model's performance, we use **K-Fold Cross-Validation**.

### How it Works
1.  Split the *entire* dataset into **K** equal parts (folds).
2.  Train on **K-1** folds, test on the remaining **1** fold.
3.  Repeat this **K** times, so every fold gets a turn as the test set.
4.  Average the **K** scores to get the final performance metric.

This gives us a much more robust estimate of how the model will perform on new data.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

# We'll use the 'best' depth we found (or a reasonable default like 3 or 4)
# Let's use max_depth=4 for this robust evaluation
robust_clf = DecisionTreeClassifier(max_depth=4, random_state=42)

# Set up 10-Fold Cross-Validation
# shuffle=True ensures the data order doesn't bias the splits
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Calculate scores
cv_scores = cross_val_score(robust_clf, X, y, cv=kf, scoring='accuracy')

print(f"Cross-Validation Scores (10 Folds):\n{cv_scores}")
print("-" * 30)
print(f"Mean Accuracy: {cv_scores.mean():.3f}")
print(f"Standard Deviation: {cv_scores.std():.3f}")

Validation with k-folds suggest pretty consistent performance. Notice there is variation, one attempt got 98% while another got 89%.

Something to think about:
- If our model averaged about 95% accuracy and was used in the real-world, we'd be making an incorrect breast cancer diagnosis in about 1 in 20 women who are tested.